# Telco → Dataverse Account Mapping

Transform the IBM Telco Customer Churn dataset into a Dataverse-ready DataFrame
that the loader script (Block 3A.3) will upload as Accounts.

**Inputs:** `data/raw/telco_churn.csv` (7,043 rows × 21 columns)
**Outputs:** `data/processed/accounts_for_upload.csv` (ready-to-load)

This notebook does NOT touch the Dataverse. All work is local.


## Setup

In [ ]:
import sys
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
from faker import Faker

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

SEED = 42
np.random.seed(SEED)
fake = Faker("en_US")
Faker.seed(SEED)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
print("Setup OK")


## 1. Load and inspect Telco data

In [ ]:
input_path = project_root / "data" / "raw" / "telco_churn.csv"
df_telco = pd.read_csv(input_path)
print(f"Shape: {df_telco.shape}")
print(f"Columns: {list(df_telco.columns)}")
df_telco.head()


## 2. Clean the data

Known quirk: `TotalCharges` loads as `object` because some rows have empty strings
instead of numbers. These are new customers (`tenure = 0`) who haven't been billed yet.


In [ ]:
df_telco["TotalCharges"] = pd.to_numeric(df_telco["TotalCharges"], errors="coerce")
nulls_before = df_telco["TotalCharges"].isnull().sum()
df_telco["TotalCharges"] = df_telco["TotalCharges"].fillna(0)

print(f"TotalCharges nulls coerced to 0: {nulls_before}")
print(f"Total nulls remaining: {df_telco.isnull().sum().sum()}")
print(f"\nChurn distribution:")
print(df_telco["Churn"].value_counts(normalize=True).round(3))


## 3. Mapping strategy

| Telco field | D365 Account field | Conversion |
|---|---|---|
| `customerID` | `telco_customer_id` (custom text) | direct copy |
| (none) | `name` | generated via `fake.company()` |
| `tenure` | `overriddencreatedon` | `today - tenure_months * 30.44 days` |
| `tenure` | `tenure_months` (custom int) | direct copy |
| `MonthlyCharges` | `revenue` | `monthly * 12` (annualized) |
| `TotalCharges` | `total_charges` (custom decimal) | direct copy |
| `Contract` | `contract_type` (custom text) | direct copy |
| `InternetService` | `internet_service` (custom text) | direct copy |
| `TechSupport` | `tech_support` (custom bool) | `Yes` → True, else False |
| `OnlineSecurity` | `online_security` (custom bool) | same |
| `PaymentMethod` | `payment_method` (custom text) | direct copy |
| **`Churn`** | **`statecode` / `statuscode`** | **Yes → (1, 2) Inactive, No → (0, 1) Active** |

**Why this set:** keeps the features with highest predictive importance from typical Telco churn analyses (tenure, contract, charges, internet service, tech support). Drops demographic features (gender, SeniorCitizen, Partner, Dependents) — weak signal in churn prediction and not relevant to B2B Account modeling.

**Why `overriddencreatedon` instead of `createdon`:** `createdon` is a system field that Dataverse sets to NOW automatically on insert. `overriddencreatedon` is the canonical field for **data migration** scenarios where you want to preserve the historical creation date.


## 4. Build the mapping

In [ ]:
def yes_no_to_bool(value: str) -> bool:
    return value == "Yes"


def tenure_to_iso_date(tenure_months: int, reference: datetime) -> str:
    days_back = int(tenure_months * 30.44)
    return (reference - timedelta(days=days_back)).isoformat()


today = datetime.now()
print(f"Reference 'today' for backdating createdon: {today.isoformat()}")


In [ ]:
df_accounts = pd.DataFrame({
    # Standard Dataverse Account fields
    "name": [fake.company() for _ in range(len(df_telco))],
    "revenue": (df_telco["MonthlyCharges"] * 12).round(2),
    "overriddencreatedon": df_telco["tenure"].apply(lambda t: tenure_to_iso_date(t, today)),
    "statecode": (df_telco["Churn"] == "Yes").astype(int),
    "statuscode": (df_telco["Churn"] == "Yes").astype(int) + 1,
    # Custom fields (will be prefixed e.g. rm_telco_customer_id on actual upload)
    "telco_customer_id": df_telco["customerID"],
    "tenure_months": df_telco["tenure"].astype(int),
    "total_charges": df_telco["TotalCharges"].round(2),
    "contract_type": df_telco["Contract"],
    "internet_service": df_telco["InternetService"],
    "tech_support": df_telco["TechSupport"].apply(yes_no_to_bool),
    "online_security": df_telco["OnlineSecurity"].apply(yes_no_to_bool),
    "payment_method": df_telco["PaymentMethod"],
})

print(f"Shape: {df_accounts.shape}")
df_accounts.head()


## 5. Validate the mapping

In [ ]:
print("=== Null counts ===")
print(df_accounts.isnull().sum())
print()
print("=== Dtypes ===")
print(df_accounts.dtypes)
print()
print("=== Target (statecode) distribution ===")
print(df_accounts["statecode"].value_counts(normalize=True).round(3))
print()
print("=== Revenue range ===")
print(f"min={df_accounts['revenue'].min():.2f} | max={df_accounts['revenue'].max():.2f} | mean={df_accounts['revenue'].mean():.2f}")
print()
print("=== overriddencreatedon range ===")
print(f"oldest: {df_accounts['overriddencreatedon'].min()}")
print(f"newest: {df_accounts['overriddencreatedon'].max()}")
print()
print("=== Sample churner ===")
print(df_accounts[df_accounts["statecode"] == 1].iloc[0])


## 6. Export for the loader (Block 3A.3)

In [ ]:
output_path = project_root / "data" / "processed" / "accounts_for_upload.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
df_accounts.to_csv(output_path, index=False)

size_kb = output_path.stat().st_size / 1024
print(f"✅ Saved: {output_path}")
print(f"   Rows: {len(df_accounts):,}")
print(f"   Size: {size_kb:.1f} KB")
print()
print("Next: Block 3A.2 (create custom fields in Power Apps maker)")
print("Then: Block 3A.3 (loader script that POSTs these rows to Dataverse)")
